> **Data note:** this notebook was run on a private clinical cohort that is not included in this repository, so it will not run as-is. The public diabetes dataset used in the paper is in `data/`; see `03_rail_d.ipynb` and `04_c_rail_d.ipynb` for runnable versions.

In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

from dotenv import load_dotenv, find_dotenv
import requests
import os

from pydantic import BaseModel, Field
from typing import Literal
from langchain.output_parsers import PydanticOutputParser
from langchain.prompts import ChatPromptTemplate

from tqdm import tqdm

from langchain_openai import ChatOpenAI
import json

from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, accuracy_score

from typing import Tuple, Union, Dict

In [ ]:

@dataclass
class KNNState:
    df: pd.DataFrame
    feature_cols: List[str]
    medians: pd.Series
    scaler: StandardScaler
    feats_imputed: np.ndarray
    X_scaled: np.ndarray
    nn: NearestNeighbors
    excel_offset: int
    # NEW: class-specific subsets
    label_col: str
    pos_indices: np.ndarray         # global row indices where IEC_HS == 1
    neg_indices: np.ndarray         # global row indices where IEC_HS == 0
    pos_X: Optional[np.ndarray]     # X_scaled for IEC_HS == 1
    neg_X: Optional[np.ndarray]     # X_scaled for IEC_HS == 0
    pos_nn: Optional[NearestNeighbors]
    neg_nn: Optional[NearestNeighbors]

In [ ]:
def build_state(
    df: pd.DataFrame,
    use_cols: Optional[List[str]] = None,
    excel_header_on_row1: bool = True,
) -> KNNState:
    # --- features as before ---
    feature_cols = use_cols if use_cols is not None else list(df.columns[:24])

    feats = df[feature_cols].apply(pd.to_numeric, errors="coerce")
    medians = feats.median(numeric_only=True)
    feats = feats.fillna(medians)
    feats_imputed = feats.values

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(feats_imputed)

    nn = NearestNeighbors(metric="euclidean")
    nn.fit(X_scaled)

    # --- class-specific neighbors (IEC_HS required) ---
    label_col = "IEC_HS"
    if label_col not in df.columns:
        raise ValueError(f"Required label column '{label_col}' not found in DataFrame.")

    labels = df[label_col].values
    pos_mask = (labels == 1)
    neg_mask = (labels == 0)

    pos_indices = np.where(pos_mask)[0]
    neg_indices = np.where(neg_mask)[0]

    pos_X = X_scaled[pos_indices] if len(pos_indices) > 0 else None
    neg_X = X_scaled[neg_indices] if len(neg_indices) > 0 else None

    pos_nn = None
    neg_nn = None
    if pos_X is not None and len(pos_indices) > 0:
        pos_nn = NearestNeighbors(metric="euclidean")
        pos_nn.fit(pos_X)
    if neg_X is not None and len(neg_indices) > 0:
        neg_nn = NearestNeighbors(metric="euclidean")
        neg_nn.fit(neg_X)

    excel_offset = 2 if excel_header_on_row1 else 1
    return KNNState(
        df=df,
        feature_cols=feature_cols,
        medians=medians,
        scaler=scaler,
        feats_imputed=feats_imputed,
        X_scaled=X_scaled,
        nn=nn,
        excel_offset=excel_offset,
        label_col=label_col,
        pos_indices=pos_indices,
        neg_indices=neg_indices,
        pos_X=pos_X,
        neg_X=neg_X,
        pos_nn=pos_nn,
        neg_nn=neg_nn,
    )

In [ ]:
#find the closest neighbors

def closest_indices_for_row(
    row_idx: int,
    state: KNNState,
    k: int = 5,
    return_distances: bool = False,
) -> Union[
    Tuple[np.ndarray, np.ndarray],                                     # (pos_inds, neg_inds)
    Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray],             # (pos_inds, pos_dists, neg_inds, neg_dists)
    Dict[str, np.ndarray]
]:
    """
    Return the k closest neighbors for the given row, separately for:
      - IEC_HS == 1  (positives)
      - IEC_HS == 0  (negatives)
    Excludes the row itself when it belongs to that class subset.
    """
    q = state.X_scaled[row_idx].reshape(1, -1)

    def _query_subset(sub_nn, sub_indices, k, drop_self: bool) -> Tuple[np.ndarray, np.ndarray]:
        if sub_nn is None or len(sub_indices) == 0:
            return np.array([], dtype=int), np.array([], dtype=float)

        # if this row belongs to the subset, ask for k+1 and drop self
        n_needed = min(k + (1 if drop_self else 0), len(sub_indices))
        distances, indices = sub_nn.kneighbors(q, n_neighbors=n_needed)
        inds_sub = indices[0].tolist()
        dists = distances[0].tolist()

        # map subset indices -> global row indices
        global_inds = [int(sub_indices[i]) for i in inds_sub]

        if drop_self and row_idx in global_inds:
            j = global_inds.index(row_idx)
            global_inds.pop(j)
            dists.pop(j)

        # trim to k
        global_inds = np.array(global_inds[:k], dtype=int)
        dists = np.array(dists[:k], dtype=float)
        return global_inds, dists

    # Determine if the query row is in each subset
    in_pos = row_idx in set(state.pos_indices.tolist())
    in_neg = row_idx in set(state.neg_indices.tolist())

    pos_inds, pos_dists = _query_subset(state.pos_nn, state.pos_indices, k, drop_self=in_pos)
    neg_inds, neg_dists = _query_subset(state.neg_nn, state.neg_indices, k, drop_self=in_neg)

    if return_distances:
        return pos_inds, pos_dists, neg_inds, neg_dists
    else:
        return pos_inds, neg_inds

In [ ]:
def pretty_print_row(row_idx: int, state: KNNState):
    """Prints the full Excel row (all columns) for the given row index."""
    row_df = pd.DataFrame([state.df.iloc[row_idx]], columns=state.df.columns)
    print(row_df.to_string(index=False))

In [ ]:
#format the few shot examples

def format_fewshot(row_idx: int, state: KNNState) -> dict:
    """
    Return a row as a few-shot example object:
    - features: dict of columns A..X
    - label: value from column Y
    - rationale: dict with AB, AC, AD
    """
    # Identify relevant columns
    feature_cols = list(state.df.columns[:24])      # A..X
    label_col = state.df.columns[24]               # Y
    #reasoning_cols = list(state.df.columns[27:30]) # AB, AC, AD
    reasoning_cols = list(state.df.columns[29:30]) # AD


    row = state.df.iloc[row_idx]

    return {
        "patient_id": row_idx,
        "features": row[feature_cols].to_dict(),
        #"rationale": {col: row[col] for col in reasoning_cols},
        "label": row[label_col]
    }

In [ ]:
def render_few_shots_block(df: pd.DataFrame, neighbor_indices: list[int], state: KNNState) -> str:
    """
    Build a text block for the human prompt with multiple few-shot examples.
    Each example has: features JSON (A..X), IEC_HS label, and reasoning (AB/AC/AD).
    """
    lines = []
    for i, idx in enumerate(neighbor_indices, start=1):
        ex = format_fewshot(idx, state)
        lines.append(
        f"patient_id: {ex['patient_id']}\n"
        f"features_json: {ex['features']}\n"
        #f"rationale: {ex['rationale']}\n"
        f"label: {ex['label']}\n"
    )
    return "\n".join(lines).strip()

In [ ]:
#all configurations

#EXCEL_PATH = "../data/Patients_toy.xlsx"       # adjust path if neede
EXCEL_PATH = "../data/iec_hs_rationales.xlsx"
SHEET = 0                                           # or a sheet name
MODEL = "gpt-4o-mini"   # any OpenAI chat model
TEMPERATURE = 0.0                                   # deterministic classification
TARGET_COL = "IEC_HS"                               # ground-truth label in your file
OUT_CSV = "../outputs/granite3-1-8B-C-RAIL.csv"

In [ ]:
# OpenAI API access: put OPENAI_API_KEY in your environment or in a local .env file
# (see .env.example). Never commit your key.
load_dotenv(find_dotenv())
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY is not set")

In [ ]:
# strcutured output and getting the prompt

class IecHsPrediction(BaseModel):
    summary: str = Field(
        ..., description="summary of combinations of feature values that leads to IEC_HS score in the given examples. Compare the given training examples to arrive at this crisp summary."
    )
    rationale: str = Field(
        ..., description="Based on the generated summary and given patient's features, key supporting evidence for IEC_HS to be positive as well as contradicting factors. Followed by overall reasoning for what should be the prediction for IEC_HS."
    )
    iec_hs: Literal[0, 1] = Field(..., description="Predicted IEC_HS class label (0 or 1). Predict IEC_HS to be 1 when you have good confidence only.")
    confidence: float = Field(
        ..., ge=0.0, le=1.0,
        description="Model self-reported confidence between 0 and 1."
    )
   

#parser = JsonOutputParser(pydantic_object=IecHsPrediction)
parser = PydanticOutputParser(pydantic_object=IecHsPrediction)

system_msg = """You are a careful clinical classifier.  
Your task is - given a single patient's anonymized features, predict whether IEC_HS is 0 or 1.
You are also given a set of close by example patients and their IEC_HS scores.
You should first create a summary by comparing the features of given examples to determine which combination of features lead to what value of IEC_HS score.
You should then use the generated summary and the given patient's feature to generate a rationale and then predict the IEC_HS score. 


Output MUST be valid JSON matching this schema:
- summary: A summary (3-5 sentences only) of combinations of feature values that leads to IEC_HS score in given examples. Compare the given training examples to arrive at this crisp summary.
- rationale: short reasoning for prediction (3-5 sentences). Use the generated summary of the given training examples and given patient's features to come up with rationale for the patient.
- iec_hs: prediction of 0 or 1 for the given patient. Predict IEC_HS to be 1 only when you have good confidence.
- confidence: a number in [0, 1]
"""

human_template = """
### Few-shot examples
{few_shots_block}

### Patient to classify
Patient features (key → value) as JSON:
{patient_json}

Instructions:
1) Generate summary of findings given the examples containing the combinations of feature values that leads to IEC_HS score of 0 or 1. Compare the given training examples to arrive at this crisp summary.
2) Predict IEC_HS (0 or 1). Predict IEC_HS to be 1 only if you have a strong confidenc.
3) Return ONLY JSON, no extra text.

Use this format:
{format_instructions}

"""

prompt = ChatPromptTemplate.from_messages(
    [("system", system_msg), ("human", human_template)]
)

# IMPORTANT: inject the raw JSON schema as a *variable*, not as literal text
prompt = prompt.partial(format_instructions=parser.get_format_instructions())

In [ ]:
# Prompt without summary. Just like RAIL

# strcutured output and getting the prompt

class IecHsPrediction(BaseModel):
    rationale: str = Field(
        ..., description="key supporting evidence for IEC_HS to be positive as well as contradicting factors. Followed by overall reasoning for what should be the prediction for IEC_HS."
    )
    iec_hs: Literal[0, 1] = Field(..., description="Predicted IEC_HS class label (0 or 1).")
    confidence: float = Field(
        ..., ge=0.0, le=1.0,
        description="Model self-reported confidence between 0 and 1."
    )
   

#parser = JsonOutputParser(pydantic_object=IecHsPrediction)
parser = PydanticOutputParser(pydantic_object=IecHsPrediction)

system_msg = """You are a careful clinical classifier. 
Given a single patient's anonymized features, predict whether IEC_HS = 0 or 1.
Only use the provided features—do not invent data.

Output MUST be valid JSON matching this schema:
- rationale: short reasoning for prediction (3-5 sentences)
- iec_hs: 0 or 1
- confidence: a number in [0, 1]
"""

human_template = """
### Few-shot examples
{few_shots_block}

### Patient to classify
Patient features (key → value) as JSON:
{patient_json}

Instructions:
1) Consider all the supporting as well as contradicting evidences and reason about what the prediction of IEC_HS should be.
2) Predict IEC_HS (0 or 1).
3) Return ONLY JSON, no extra text.

Use this format:
{format_instructions}

Output MUST be valid JSON matching this schema:
- rationale: short reasoning for prediction (3-5 sentences)
- iec_hs: 0 or 1
- confidence: a number in [0, 1]
"""

prompt = ChatPromptTemplate.from_messages(
    [("system", system_msg), ("human", human_template)]
)

# IMPORTANT: inject the raw JSON schema as a *variable*, not as literal text
prompt = prompt.partial(format_instructions=parser.get_format_instructions())

In [ ]:
llm = ChatOpenAI(
        model=MODEL,
        temperature=0,
        max_retries=2,
        max_tokens=4096,
        api_key=api_key,
    )

In [ ]:
chain = prompt | llm | parser

In [ ]:
df = pd.read_excel("../data/iec_hs_rationales.xlsx")
selected_cols = ["CRP", "Diagnosis", "Ferritin", "Tocilizumab given"]
state = build_state(df, use_cols=selected_cols)
#state = build_state(df)  # uses first 24 columns (A..X) by default
test_row = 0
print(test_row)
pos_inds, pos_dists, neg_inds, neg_dists = closest_indices_for_row(test_row, state, k=3, return_distances=True)
nn_inds = np.concatenate([pos_inds, neg_inds])
all_dists = np.concatenate([pos_dists, neg_dists])
order = np.argsort(all_dists)[::-1]
nn_inds_sorted = nn_inds[order]
print(render_few_shots_block(df, nn_inds_sorted, state))

In [ ]:
results = []
errored_rows = []

state = build_state(df)  # uses first 24 columns (A..X) by default


#selected_cols = ["CRP", "Diagnosis", "Ferritin", "Tocilizumab given"]

#state = build_state(df, use_cols=selected_cols)

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Scoring patients"):
    # --- everything before model call runs once ---
    pos_inds, pos_dists, neg_inds, neg_dists = closest_indices_for_row(
        idx, state, k=3, return_distances=True
    )
    nn_inds = np.concatenate([pos_inds, neg_inds])
    all_dists = np.concatenate([pos_dists, neg_dists])
    order = np.argsort(all_dists)[::-1]
    nn_inds_sorted = nn_inds[order]

    few_shot_block = render_few_shots_block(df, nn_inds_sorted, state)
    record = format_fewshot(idx, state)
    patient_features = record["features"]

    # safety
    assert TARGET_COL not in patient_features

    # --- retry loop ONLY around the model call ---
    success = False
    last_error = None
    for attempt in range(5):
        try:
            pred: IecHsPrediction = chain.invoke({
                "few_shots_block": few_shot_block,
                "patient_json": json.dumps(patient_features, ensure_ascii=False)
            })

            print(f"prediction for row {idx} is {pred.iec_hs} with confidence {pred.confidence} and true lable is {row.get(TARGET_COL, None)}")
            #print(f"summary for row {idx} is {pred.summary}")
            #print(f"rationale for row {idx} is {pred.rationale}")

            results.append({
                "row_index": idx,
                "iec_hs_pred": int(pred.iec_hs),
                "confidence": float(pred.confidence),
                "rationale": pred.rationale,
                "iec_hs_true": row.get(TARGET_COL, None)
            })

            success = True
            break  # exit retry loop if successful
        except Exception as e:
            last_error = e
            print(f"Attempt {attempt+1} failed for row {idx}: {e}")
            print(f"output is {pred}")
            continue  # try again

    if not success:
        errored_rows.append((idx, str(last_error)))
        print(f"prediction for row {idx} is SKIPPED after 3 attempts")

In [ ]:
out_df = pd.DataFrame(results)

# Keep only rows where both are non-null
valid = out_df["iec_hs_pred"].notna() & out_df["iec_hs_true"].notna()
y_true = out_df.loc[valid, "iec_hs_true"].astype(int)
y_pred = out_df.loc[valid, "iec_hs_pred"].astype(int)

# Overall metrics (positive class = 1)
accuracy   = accuracy_score(y_true, y_pred)
precision1 = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
recall1    = recall_score(y_true, y_pred,    pos_label=1, zero_division=0)
f1_pos     = f1_score(y_true, y_pred,        pos_label=1, zero_division=0)

# Negative-class metrics (label = 0)
precision0 = precision_score(y_true, y_pred, pos_label=0, zero_division=0)
recall0    = recall_score(y_true, y_pred,    pos_label=0, zero_division=0)
f1_neg     = f1_score(y_true, y_pred,        pos_label=0, zero_division=0)

# Macro F1 (unweighted mean of per-class F1s)
f1_macro   = f1_score(y_true, y_pred, average="macro", zero_division=0)

print(f"Accuracy:     {accuracy:.3f}")
print(f"Pos P/R/F1:   P={precision1:.3f}  R={recall1:.3f}  F1={f1_pos:.3f}")
print(f"Neg P/R/F1:   P={precision0:.3f}  R={recall0:.3f}  F1={f1_neg:.3f}")
print(f"Macro F1:     {f1_macro:.3f}")

# Confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
print("\nConfusion Matrix (rows = true, cols = pred):")
print("         Pred 0   Pred 1")
print(f"True 0   {cm[0,0]:6}   {cm[0,1]:6}")
print(f"True 1   {cm[1,0]:6}   {cm[1,1]:6}")

In [ ]:
out_df = pd.DataFrame(results)
print(out_df.columns.tolist())
# Optional: simple accuracy if labels exist
if TARGET_COL in df.columns and out_df["iec_hs_pred"].notna().any():

    acc = (out_df["iec_hs_pred"] == out_df["iec_hs_true"]).mean()
    print(f"Accuracy on labeled rows: {acc:.3f}")

#print(out_df)
out_df.to_csv("../outputs/llama3.1-8b-C-RAIL.csv", index=False)

In [ ]:
len(errored_rows)